# Managing projections with GeoMakie

## The packages
We will rely essentialy on the plotting library [Makie](https://makie.org/website/) and on [GeoMakie](https://geo.makie.org/stable/), to display information on a given map projection.    
Additionaly we will load [NCDatasets](https://github.com/JuliaGeo/NCDatasets.jl) to load some data.        

There are several backends. For instance:
- if you want to work OpenGL, then you need to load `GLMakie`
- if you want to render 2D by Cairo Graphics, you will use `CairoMakie`.

In [1]:
using Pkg
Pkg.activate("../")
Pkg.instantiate()
using Makie
using CairoMakie
using GeoMakie
using NCDatasets
using JupyterFormatter
using Glob
enable_autoformat()

  Activating project at `~/Projects/maps-with-makie`
Precompiling packages...
   0.4 s  ✓ IfElse
   0.4 s  ✓ CustomUnitRanges
   0.4 s  ✓ UnPack
   0.4 s  ✓ SIMDTypes
   0.5 s  ✓ Interfaces
   0.6 s  ✓ FieldMetadata
   0.6 s  ✓ ManualMemory
   0.6 s  ✓ Quaternions
   0.7 s  ✓ IterTools
   0.7 s  ✓ ArrayInterface
   0.7 s  ✓ ComputationalResources
   0.8 s  ✓ Distances
   0.5 s  ✓ Missings
   0.5 s  ✓ NaNMath
   0.5 s  ✓ MapTiles
   1.0 s  ✓ ProgressMeter
   1.4 s  ✓ FillArrays
   1.0 s  ✓ Setfield
   0.7 s  ✓ CommonWorldInvalidations
   1.7 s  ✓ IrrationalConstants
   1.0 s  ✓ ComputePipeline
   0.9 s  ✓ Fmt_jll
   1.8 s  ✓ ColorTypes
   1.3 s  ✓ CpuId
   1.6 s  ✓ ChainRulesCore
   1.7 s  ✓ StructUtils
   2.2 s  ✓ DataStructures
   0.9 s  ✓ Kerberos_krb5_jll
   0.8 s  ✓ rapidjson_jll
   1.5 s  ✓ ICU_jll
   0.9 s  ✓ LERC_jll
   1.0 s  ✓ FriBidi_jll
   1.5 s  ✓ Qhull_jll
   1.1 s  ✓ brotli_jll
   0.9 s  ✓ SQLite_jll
   2.6 s  ✓ Accessors
   0.9 s  ✓ snappy_jll
   1.0 s  ✓ libdeflate_jll


1-element Vector{Function}:
 format_current_cell (generic function with 1 method)

## Data
We will load the GEBCO bathymetry.

In [50]:
topofile = "./gebco_30sec_16.nc"

NN = 1
ds = NCDataset(topofile)
lonb = ds["lon"][1:NN:end]
latb = ds["lat"][1:NN:end]
bathy = ds["bat"][1:NN:end, 1:NN:end]
close(ds)
@info(size(bathy));

theyear = "2020"
datadir = "/home/ctroupin/Dataz/EN/"
datafilelist = Glob.glob("*$(theyear)*.nc", datadir)

[ Info: (2700, 1352)


12-element Vector{String}:
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202001.nc"
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202002.nc"
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202003.nc"
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202004.nc"
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202005.nc"
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202006.nc"
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202007.nc"
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202008.nc"
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202009.nc"
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202010.nc"
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202011.nc"
 "/home/ctroupin/Dataz/EN/EN.4.2.2.f.profiles.g10.202012.nc"

In [4]:
nc = NCDataset(first(datafilelist))
lon = nc["LONGITUDE"][:]
lat = nc["LATITUDE"][:]
close(nc)

closed Dataset

## Working with GeoMakie
The procedure for the creation of a plot is very similar, the main difference is that during the creation of the Axis (which is called here `GeoAxis`), one has to define a projection.
### Creating a projection
There are different ways to create a projection. They are defined as _proj-string_ (see https://proj.org/en/stable/operations/projections/index.html).
#### Global projections
We present a few examples using global projections

In [43]:
globalproj1 = "+proj=ortho +lon_0=-160 +lat_0=-20"
using ColorSchemes
topocmap = ColorSchemes.delta

In [55]:
topocmap2 = Makie.to_colormap(:delta)[80:end-80]

In [57]:
f = Figure(size = (1200, 1200))
ax = GeoAxis(f[1, 1], dest = globalproj1, title = "", xticks = [40.0])

hm = surface!(
    ax,
    lonb,
    latb,
    bathy,
    colormap = topocmap,
    colorrange = [-5000, 5000],
    shading = NoShading,
    interpolate = true,
)
contour!(ax, lonb, latb, bathy, levels = [0.0], color = :black)

for datafile in datafilelist
    nc = NCDataset(first(datafilelist))
    lon = nc["LONGITUDE"][:]
    lat = nc["LATITUDE"][:]
    close(nc)
    scatter!(ax, lon, lat, markersize = 4, color = :white)
end
#lines!(ax, GeoMakie.coastlines(10), color = :grey)
save("/home/ctroupin/EN4_obs_$(theyear).png", f)